<div style="font-family: 'Helvetica Neue', Arial, sans-serif; background:#0F2E2B; padding: 40px 44px; border-radius: 14px; margin-bottom: 8px; position:relative; overflow:hidden;">
  <div style="font-size:150px; font-weight:900; color:rgba(255,255,255,0.04); position:absolute; top:-30px; right:24px; line-height:1; letter-spacing:-0.05em;">04</div>
  <div style="font-size:10px; color:#5FCDAA; letter-spacing:0.3em; text-transform:uppercase; margin-bottom:18px;">NOVA IMS &middot; 2025/2026</div>
  <div style="font-size:38px; font-weight:800; color:#F5F3EC; letter-spacing:-0.02em; line-height:1.1; margin-bottom:8px;">Home Credit <span style="color:#5FCDAA;">Default Risk.</span></div>
  <div style="font-size:12px; color:#E08A6D; font-weight:500; margin-bottom:28px; letter-spacing:0.05em;">Notebook 4 &mdash; Modeling: Data Split, Model Selection &amp; Training</div>
  <div style="display:flex; gap:56px;">
    <div>
      <div style="font-size:9px; color:#5FCDAA; letter-spacing:0.2em; text-transform:uppercase; margin-bottom:8px;">Group 1</div>
      <div style="font-size:11px; color:#C5D8D2; line-height:1.95;">Alexandra Varela, 20250514<br>Francisca Fernandes, 20250406<br>Mariana Melo, 20250414<br>Rui Ferreira, 20250473<br>Tiago Antunes, 20250357</div>
    </div>
    <div>
      <div style="font-size:9px; color:#5FCDAA; letter-spacing:0.2em; text-transform:uppercase; margin-bottom:8px;">Course</div>
      <div style="font-size:11px; color:#C5D8D2; line-height:1.95;">MLOps<br>MSc Data Science &amp; Advanced Analytics<br>NOVA Information Management School</div>
    </div>
  </div>
</div>

## Table of Contents

1. [Setup & Imports](#setup)
2. [Load Engineered Data](#load-data)
3. [C1 — Data Split: Stratified Train / Validation](#c1-data-split)
4. [C2 — Model Selection: GridSearchCV with MLflow](#c2-gridsearch)
5. [C3 — Model Selection: Optuna TPE Search](#c3-optuna)
6. [C4 — Choose Best & Compare](#c4-compare)
7. [C5 — Final Model Training](#c5-final-train)
8. [C6 — Decision Threshold Selection: Cost-Sensitive](#c6-threshold)
9. [C7 — Model Evaluation: Credit Scoring Metrics](#c7-evaluation)
10. [C8 — SHAP Explanations](#c8-shap)
11. [C9 — MLflow Model Registry](#c9-registry)
12. [Summary](#summary)

## About This Notebook

This notebook prototypes **Part C** of the Home Credit Default Risk MLOps pipeline:

- **Stratified data split** — ensuring balanced class representation across train/validation/test sets and preventing target leakage
- **Dual model search** — exhaustive `GridSearchCV` (Week 3 lecture content) followed by Bayesian optimisation with `Optuna` TPE sampler (Week 5 lecture content)
- **Final model training** with optional probability calibration via `CalibratedClassifierCV` (isotonic regression), converting raw scores into well-calibrated PD (Probability of Default) estimates
- **Decision threshold selection** using the cost-sensitive framework from *Verbraken et al. (2014)*, where the asymmetric cost of missing a defaulter (FN) is 10× the cost of a false alarm (FP)
- **Full credit-scoring evaluation**: ROC-AUC, Gini coefficient, KS statistic, PR-AUC, Brier score, log-loss
- **SHAP explainability** (Week 7 lecture content) — TreeExplainer for global and local feature attributions
- **MLflow Model Registry** — logging the final model artifact and promoting it to `"production"` alias

All decisions made here are implemented as Kedro pipeline nodes under `pipelines/model_selection` and `pipelines/model_train`.

<a id="setup"></a>
## 1. Setup & Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import warnings
import os
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Data / numeric ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Scikit-learn ───────────────────────────────────────────────────────────────
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, brier_score_loss, log_loss,
    confusion_matrix, ks_2samp
)
from sklearn.preprocessing import label_binarize
from sklearn.pipeline import Pipeline

# ── MLflow ─────────────────────────────────────────────────────────────────────
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

# ── Optuna ─────────────────────────────────────────────────────────────────────
import optuna
from optuna.integration.mlflow import MLflowCallback
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── SHAP ───────────────────────────────────────────────────────────────────────
import shap

print("All imports successful.")
print(f"  sklearn  : {__import__('sklearn').__version__}")
print(f"  mlflow   : {mlflow.__version__}")
print(f"  optuna   : {optuna.__version__}")
print(f"  shap     : {shap.__version__}")

In [ ]:
# ── Colour palette ─────────────────────────────────────────────────────────────
PALETTE = {
    "bg_header"  : "#0F2E2B",
    "teal"       : "#5FCDAA",
    "dark_teal"  : "#1B7A6E",
    "coral"      : "#E08A6D",
    "cream"      : "#F5F3EC",
    "plot_bg"    : "#F8F6EF",
    "watermark"  : "rgba(255,255,255,0.04)",
}

# ── Global matplotlib style ────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor"    : PALETTE["plot_bg"],
    "axes.facecolor"      : PALETTE["plot_bg"],
    "axes.edgecolor"      : "#CCCCCC",
    "axes.labelcolor"     : "#333333",
    "axes.titlesize"      : 13,
    "axes.labelsize"      : 11,
    "xtick.color"         : "#555555",
    "ytick.color"         : "#555555",
    "grid.color"          : "#DDDDDD",
    "grid.linestyle"      : "--",
    "grid.alpha"          : 0.7,
    "font.family"         : "sans-serif",
    "legend.framealpha"   : 0.85,
    "legend.fontsize"     : 9,
})

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT       = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH  = ROOT / "data"
RAW_PATH   = DATA_PATH / "01_raw"
MODEL_INPUT= DATA_PATH / "05_model_input"
REPORT_DIR = DATA_PATH / "08_reporting"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Root     : {ROOT}")
print(f"Reports  : {REPORT_DIR}")

<a id="load-data"></a>
## 2. Load Engineered Data

We attempt to load the feature-engineered outputs produced by the Kedro `feature_engineering` pipeline (saved to `data/05_model_input/`).  
If those files are not present (e.g., the pipeline has not been run yet), we generate a **synthetic dataset** that mirrors the Home Credit feature space: 5 000 observations, 25 features with domain-realistic names, and a realistic ≈ 8 % default rate.

In [ ]:
FEATURE_NAMES = [
    "AMT_CREDIT", "AMT_ANNUITY", "AMT_INCOME_TOTAL", "AMT_GOODS_PRICE",
    "DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM",
    "CNT_FAM_MEMBERS", "CNT_CHILDREN",
    "BUREAU_ACTIVE_COUNT", "BUREAU_CLOSED_COUNT", "BUREAU_AMT_CREDIT_SUM",
    "PREV_APP_COUNT", "PREV_AMT_APPLICATION_MEAN", "PREV_AMT_CREDIT_MEAN",
    "INSTAL_AMT_PAYMENT_SUM", "INSTAL_DAYS_PAST_DUE_MAX", "POS_MONTHS_BALANCE_MIN",
]

try:
    X_all = pd.read_parquet(MODEL_INPUT / "X_train_data.parquet")
    y_all = pd.read_parquet(MODEL_INPUT / "y_train_data.parquet").squeeze()
    print("Loaded pipeline outputs from data/05_model_input/")
    DATA_SOURCE = "pipeline"
except Exception:
    print("Pipeline outputs not found — generating synthetic dataset.")
    DATA_SOURCE = "synthetic"

    X_raw, y_raw = make_classification(
        n_samples=5000,
        n_features=25,
        n_informative=12,
        n_redundant=5,
        n_clusters_per_class=2,
        weights=[0.92, 0.08],   # ≈ 8 % default rate
        flip_y=0.01,
        random_state=42,
    )
    X_all = pd.DataFrame(X_raw, columns=FEATURE_NAMES)
    y_all = pd.Series(y_raw, name="TARGET")

    # Make the credit-amount feature realistic (positive right-skewed values)
    X_all["AMT_CREDIT"]       = np.abs(X_all["AMT_CREDIT"])       * 200_000 + 50_000
    X_all["AMT_INCOME_TOTAL"] = np.abs(X_all["AMT_INCOME_TOTAL"]) * 50_000  + 30_000
    X_all["EXT_SOURCE_1"]     = (X_all["EXT_SOURCE_1"] - X_all["EXT_SOURCE_1"].min()) / \
                                 (X_all["EXT_SOURCE_1"].max() - X_all["EXT_SOURCE_1"].min())
    X_all["EXT_SOURCE_2"]     = (X_all["EXT_SOURCE_2"] - X_all["EXT_SOURCE_2"].min()) / \
                                 (X_all["EXT_SOURCE_2"].max() - X_all["EXT_SOURCE_2"].min())
    X_all["EXT_SOURCE_3"]     = (X_all["EXT_SOURCE_3"] - X_all["EXT_SOURCE_3"].min()) / \
                                 (X_all["EXT_SOURCE_3"].max() - X_all["EXT_SOURCE_3"].min())

print(f"\nDataset source : {DATA_SOURCE}")
print(f"Shape          : {X_all.shape}")
print(f"Class balance  : {y_all.value_counts().to_dict()}")
print(f"Default rate   : {y_all.mean():.2%}")

In [ ]:
X_all.describe().T.style.background_gradient(cmap="YlGn").format("{:.2f}")

<a id="c1-data-split"></a>
## C1 — Data Split: Stratified Train / Validation

**Why split before any further processing?**  
Performing transformations (imputation means, scaling statistics, target encoding) on the combined dataset would introduce **target leakage** — the validation set would "bleed" information into the training set.  The correct order is:

1. Stratified split → `(X_train, y_train)` and `(X_val, y_val)`
2. Fit all preprocessing on `X_train` only
3. Transform both train and validation using the fitted transformers

We use `stratify=y` to maintain the same minority-class proportion in every split.

In [ ]:
# ── 70 / 15 / 15 stratified split ─────────────────────────────────────────────
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all, y_all,
    test_size=0.15,
    stratify=y_all,
    random_state=42,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.15 / 0.85,   # makes val ≈ 15 % of total
    stratify=y_train_full,
    random_state=42,
)

print("Split sizes:")
print(f"  Train      : {X_train.shape[0]:>5}  (default rate {y_train.mean():.3%})")
print(f"  Validation : {X_val.shape[0]:>5}  (default rate {y_val.mean():.3%})")
print(f"  Test       : {X_test.shape[0]:>5}  (default rate {y_test.mean():.3%})")
print(f"  Total      : {X_all.shape[0]:>5}  (default rate {y_all.mean():.3%})")

In [ ]:
# ── Visualise class balance preservation ──────────────────────────────────────
split_names  = ["Original", "Train", "Validation", "Test"]
split_series = [y_all, y_train, y_val, y_test]

pos_rates = [s.mean()         for s in split_series]
neg_rates = [1 - s.mean()     for s in split_series]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Class Distribution Across Splits", fontsize=14, fontweight="bold",
             color="#222222", y=1.01)

x = np.arange(len(split_names))
width = 0.35

# Stacked bar
ax = axes[0]
ax.bar(x, neg_rates, color=PALETTE["teal"],  label="Non-default (0)", width=0.55)
ax.bar(x, pos_rates, color=PALETTE["coral"], label="Default (1)",     width=0.55,
       bottom=neg_rates)
ax.set_xticks(x)
ax.set_xticklabels(split_names)
ax.set_ylabel("Proportion")
ax.set_title("Stacked Class Proportions")
ax.legend()
ax.yaxis.grid(True)
ax.set_axisbelow(True)

# Default rate detail
ax2 = axes[1]
bars = ax2.bar(split_names, [r * 100 for r in pos_rates],
               color=[PALETTE["dark_teal"], PALETTE["teal"],
                      PALETTE["coral"], PALETTE["bg_header"]],
               edgecolor="white", linewidth=1.2)
for bar, rate in zip(bars, pos_rates):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.1,
             f"{rate:.2%}",
             ha="center", va="bottom", fontsize=10, fontweight="bold")
ax2.set_ylabel("Default Rate (%)")
ax2.set_title("Default Rate per Split")
ax2.yaxis.grid(True)
ax2.set_axisbelow(True)

plt.tight_layout()
fig.savefig(REPORT_DIR / "c1_class_distribution.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved → data/08_reporting/c1_class_distribution.png")

<a id="c2-gridsearch"></a>
## C2 — Model Selection: GridSearchCV with MLflow

### Strategy: Exhaustive Grid Search (Week 3)

For a first pass we run an **exhaustive grid search** over two model families:

| Family              | Rationale |
|---------------------|-----------|
| `RandomForest`      | Low variance, natural feature importance, parallelisable |
| `GradientBoosting`  | High capacity, sequential error correction, state-of-the-art on tabular data |

Each combination is evaluated using `StratifiedKFold(n_splits=4)` — stratification is essential because the data are imbalanced (≈ 8 % default).  
The optimisation metric is **ROC-AUC** (area under the receiver operating characteristic curve), the industry standard for binary credit-scoring tasks.

Every run is tracked in **MLflow** stored in a local SQLite database.

In [ ]:
# ── MLflow setup ───────────────────────────────────────────────────────────────
MLFLOW_DB  = str(ROOT / "mlflow_notebook.db")
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
mlflow.set_experiment("home_credit_notebook_04")

print(f"MLflow tracking URI : sqlite:///{MLFLOW_DB}")
print(f"Experiment          : home_credit_notebook_04")

In [ ]:
# ── Parameter grids (kept small for notebook speed) ────────────────────────────
param_grids = {
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
        "grid" : {
            "n_estimators"  : [100, 200],
            "max_depth"     : [6, 10, None],
            "min_samples_split": [2, 10],
        },
    },
    "GradientBoosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "grid" : {
            "n_estimators"  : [100, 200],
            "learning_rate" : [0.05, 0.1],
            "max_depth"     : [3, 5],
        },
    },
}

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

grid_results = {}

for family_name, cfg in param_grids.items():
    print(f"\nRunning GridSearchCV for {family_name} …")
    gs = GridSearchCV(
        estimator=cfg["model"],
        param_grid=cfg["grid"],
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1,
        verbose=0,
        refit=True,
    )
    gs.fit(X_train, y_train)

    best_auc   = gs.best_score_
    best_params= gs.best_params_
    val_auc    = roc_auc_score(y_val, gs.predict_proba(X_val)[:, 1])

    grid_results[family_name] = {
        "gs"          : gs,
        "cv_auc"      : best_auc,
        "val_auc"     : val_auc,
        "best_params" : best_params,
    }

    # Log to MLflow
    with mlflow.start_run(run_name=f"gridsearch_{family_name}"):
        mlflow.log_param("model_family", family_name)
        mlflow.log_param("search_method", "GridSearchCV")
        for k, v in best_params.items():
            mlflow.log_param(k, v)
        mlflow.log_metric("cv_roc_auc",  best_auc)
        mlflow.log_metric("val_roc_auc", val_auc)

    print(f"  Best CV  AUC : {best_auc:.4f}")
    print(f"  Best Val AUC : {val_auc:.4f}")
    print(f"  Best params  : {best_params}")

In [ ]:
# ── Visualise GridSearch results ───────────────────────────────────────────────
families = list(grid_results.keys())
cv_aucs  = [grid_results[f]["cv_auc"]  for f in families]
val_aucs = [grid_results[f]["val_auc"] for f in families]

fig, ax = plt.subplots(figsize=(9, 4))
y_pos  = np.arange(len(families))
width  = 0.35

bars1 = ax.barh(y_pos - width / 2, cv_aucs,  width, label="CV ROC-AUC",
                color=PALETTE["teal"],      edgecolor="white")
bars2 = ax.barh(y_pos + width / 2, val_aucs, width, label="Val ROC-AUC",
                color=PALETTE["coral"],     edgecolor="white")

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.4f}", va="center", ha="left", fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(families, fontsize=11)
ax.set_xlabel("ROC-AUC")
ax.set_title("GridSearchCV — Best ROC-AUC per Model Family", fontweight="bold")
ax.legend()
ax.set_xlim(0.5, 1.0)
ax.xaxis.grid(True)
ax.set_axisbelow(True)
plt.tight_layout()
fig.savefig(REPORT_DIR / "c2_gridsearch_auc.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved → data/08_reporting/c2_gridsearch_auc.png")

In [ ]:
# ── Print best GridSearch result ────────────────────────────────────────────────
best_gs_family = max(grid_results, key=lambda f: grid_results[f]["val_auc"])
best_gs_info   = grid_results[best_gs_family]

print("Best GridSearch result:")
print(f"  Model family : {best_gs_family}")
print(f"  CV  AUC      : {best_gs_info['cv_auc']:.4f}")
print(f"  Val AUC      : {best_gs_info['val_auc']:.4f}")
print(f"  Best params  : {best_gs_info['best_params']}")

<a id="c3-optuna"></a>
## C3 — Model Selection: Optuna TPE Search

### Bayesian Optimisation vs Grid Search (Week 5)

Exhaustive grid search evaluates **every combination** in the Cartesian product — efficient for small grids but exponentially expensive as dimensionality grows.  

**Optuna** implements **Tree-structured Parzen Estimator (TPE)** — a sequential model-based optimisation (SMBO) algorithm that:

1. Maintains a probabilistic model of the objective surface
2. Samples the next hyperparameter set from regions that are *expected to improve* the objective (exploitation) while occasionally exploring new areas (exploration)
3. Converges to good solutions in far fewer evaluations than a grid

This makes Optuna the preferred strategy when the hyperparameter space is large or continuous.

In [ ]:
# ── Optuna objective function ──────────────────────────────────────────────────
def objective(trial: optuna.Trial) -> float:
    model_type = trial.suggest_categorical("model_type", ["RF", "GBM"])

    if model_type == "RF":
        model = RandomForestClassifier(
            n_estimators     = trial.suggest_int("n_estimators",       50, 300, step=50),
            max_depth        = trial.suggest_int("max_depth",           3, 15),
            min_samples_split= trial.suggest_int("min_samples_split",   2, 20),
            max_features     = trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            class_weight     = "balanced",
            random_state     = 42,
            n_jobs           = -1,
        )
    else:
        model = GradientBoostingClassifier(
            n_estimators  = trial.suggest_int(  "n_estimators",   50, 300, step=50),
            learning_rate = trial.suggest_float("learning_rate",  0.01, 0.3, log=True),
            max_depth     = trial.suggest_int(  "max_depth",       2, 8),
            subsample     = trial.suggest_float("subsample",      0.6, 1.0),
            random_state  = 42,
        )

    auc_scores = cross_val_score(
        model, X_train, y_train,
        cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=42),
        scoring="roc_auc",
        n_jobs=-1,
    )
    return auc_scores.mean()

print("Objective function defined.")

In [ ]:
# ── Run Optuna study (20 trials) ───────────────────────────────────────────────
mlflc = MLflowCallback(
    tracking_uri=f"sqlite:///{MLFLOW_DB}",
    metric_name="cv_roc_auc",
    create_experiment=False,
    mlflow_kwargs={"experiment_id": mlflow.get_experiment_by_name(
        "home_credit_notebook_04").experiment_id},
)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="home_credit_tpe",
)

study.optimize(objective, n_trials=20, callbacks=[mlflc], show_progress_bar=False)

print(f"\nOptuna study complete.")
print(f"  Best trial  : #{study.best_trial.number}")
print(f"  Best CV AUC : {study.best_value:.4f}")
print(f"  Best params : {study.best_params}")

In [ ]:
# ── Optimisation history plot ──────────────────────────────────────────────────
trials_df = study.trials_dataframe()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Optuna TPE — Hyperparameter Search", fontsize=14,
             fontweight="bold", color="#222222")

# Left: optimisation history
ax = axes[0]
ax.scatter(trials_df["number"], trials_df["value"],
           color=PALETTE["teal"], alpha=0.7, s=40, label="Trial AUC", zorder=3)
best_so_far = trials_df["value"].cummax()
ax.plot(trials_df["number"], best_so_far,
        color=PALETTE["coral"], linewidth=2.5, label="Best so far", zorder=4)
ax.axhline(study.best_value, color=PALETTE["dark_teal"],
           linestyle="--", linewidth=1.5, label=f"Final best ({study.best_value:.4f})")
ax.set_xlabel("Trial Number")
ax.set_ylabel("CV ROC-AUC")
ax.set_title("Optimisation History")
ax.legend()
ax.yaxis.grid(True)
ax.set_axisbelow(True)

# Right: model type distribution
ax2 = axes[1]
model_col = "params_model_type" if "params_model_type" in trials_df.columns else None
if model_col:
    type_counts = trials_df[model_col].value_counts()
    bars = ax2.bar(type_counts.index, type_counts.values,
                   color=[PALETTE["teal"], PALETTE["coral"]], edgecolor="white")
    for bar in bars:
        ax2.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.3,
                 str(int(bar.get_height())),
                 ha="center", va="bottom", fontsize=11)
    ax2.set_xlabel("Model Type")
    ax2.set_ylabel("# Trials")
    ax2.set_title("Trials per Model Family")
    ax2.yaxis.grid(True)
    ax2.set_axisbelow(True)
else:
    ax2.text(0.5, 0.5, "Model type column not found in trials df",
             ha="center", va="center", transform=ax2.transAxes)

plt.tight_layout()
fig.savefig(REPORT_DIR / "c3_optuna_history.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved → data/08_reporting/c3_optuna_history.png")

In [ ]:
# ── Parameter importance plot ──────────────────────────────────────────────────
try:
    importances = optuna.importance.get_param_importances(study)
    imp_keys    = list(importances.keys())
    imp_vals    = list(importances.values())

    fig, ax = plt.subplots(figsize=(9, 4))
    y_pos = np.arange(len(imp_keys))
    colors = [PALETTE["teal"] if v >= max(imp_vals) * 0.5 else PALETTE["coral"]
              for v in imp_vals]
    bars = ax.barh(y_pos, imp_vals, color=colors, edgecolor="white")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(imp_keys, fontsize=10)
    ax.set_xlabel("Importance (FAnova)")
    ax.set_title("Optuna — Hyperparameter Importance", fontweight="bold")
    ax.xaxis.grid(True)
    ax.set_axisbelow(True)
    plt.tight_layout()
    fig.savefig(REPORT_DIR / "c3_param_importance.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved → data/08_reporting/c3_param_importance.png")
except Exception as e:
    print(f"Could not compute param importance: {e}")

<a id="c4-compare"></a>
## C4 — Choose Best & Compare

We compare the best model from each search strategy to select the **champion** for final training.

In [ ]:
# ── Build comparison table ─────────────────────────────────────────────────────
# Optuna best — retrain on full train set to get val AUC
opt_params = study.best_params.copy()
opt_model_type = opt_params.pop("model_type", "RF")

if opt_model_type == "RF":
    opt_model = RandomForestClassifier(
        **{k: v for k, v in opt_params.items() if k in [
            "n_estimators", "max_depth", "min_samples_split", "max_features"
        ]},
        class_weight="balanced", random_state=42, n_jobs=-1,
    )
else:
    opt_model = GradientBoostingClassifier(
        **{k: v for k, v in opt_params.items() if k in [
            "n_estimators", "learning_rate", "max_depth", "subsample"
        ]},
        random_state=42,
    )

opt_model.fit(X_train, y_train)
opt_val_auc = roc_auc_score(y_val, opt_model.predict_proba(X_val)[:, 1])

comparison = pd.DataFrame([
    {
        "Strategy"    : "GridSearchCV",
        "Model Family": best_gs_family,
        "CV AUC"      : round(best_gs_info["cv_auc"], 4),
        "Val AUC"     : round(best_gs_info["val_auc"], 4),
        "Key Params"  : str(best_gs_info["best_params"]),
    },
    {
        "Strategy"    : "Optuna TPE",
        "Model Family": opt_model_type,
        "CV AUC"      : round(study.best_value, 4),
        "Val AUC"     : round(opt_val_auc, 4),
        "Key Params"  : str(study.best_params),
    },
])

comparison.style \
    .background_gradient(subset=["CV AUC", "Val AUC"], cmap="YlGn") \
    .set_properties(**{"font-size": "11px"}) \
    .highlight_max(subset=["CV AUC", "Val AUC"],
                   color="#c8f7d6")

In [ ]:
# ── Select winner ──────────────────────────────────────────────────────────────
gs_val  = best_gs_info["val_auc"]
opt_val = opt_val_auc

if opt_val >= gs_val:
    CHAMPION_NAME   = f"Optuna TPE — {opt_model_type}"
    CHAMPION_MODEL  = opt_model
    CHAMPION_PARAMS = study.best_params
    print(f"Winner: Optuna TPE ({opt_model_type})  Val AUC = {opt_val:.4f}")
else:
    CHAMPION_NAME   = f"GridSearchCV — {best_gs_family}"
    CHAMPION_MODEL  = best_gs_info["gs"].best_estimator_
    CHAMPION_PARAMS = best_gs_info["best_params"]
    print(f"Winner: GridSearchCV ({best_gs_family})  Val AUC = {gs_val:.4f}")

print(f"Champion: {CHAMPION_NAME}")

<a id="c5-final-train"></a>
## C5 — Final Model Training

### Probability Calibration — CalibratedClassifierCV (Week 5)

Tree-based models (Random Forests, Gradient Boosting) output **scores**, not well-calibrated **probabilities**.  For credit risk the output is interpreted as the *Probability of Default* (PD) — a regulatory quantity that feeds into IFRS 9 provisions and Basel III capital calculations.  Miscalibrated probabilities lead to incorrect provisioning.

`CalibratedClassifierCV` with `method='isotonic'` fits a **monotone step function** that maps raw scores to calibrated probabilities via cross-validated isotonic regression (Platt scaling uses a sigmoid — faster but less flexible).

In [ ]:
# ── Train champion on full train set ───────────────────────────────────────────
BASE_MODEL = type(CHAMPION_MODEL)(**CHAMPION_MODEL.get_params())
BASE_MODEL.fit(X_train_full, y_train_full)
print(f"Base model trained on {len(y_train_full)} samples.")

# ── Probability calibration ────────────────────────────────────────────────────
CALIB_MODEL = CalibratedClassifierCV(
    estimator=type(CHAMPION_MODEL)(**CHAMPION_MODEL.get_params()),
    method="isotonic",
    cv=4,
)
CALIB_MODEL.fit(X_train_full, y_train_full)
print(f"Calibrated model trained (isotonic regression, cv=4).")

In [ ]:
# ── Calibration curve (reliability diagram) ────────────────────────────────────
prob_base  = BASE_MODEL.predict_proba(X_test)[:, 1]
prob_calib = CALIB_MODEL.predict_proba(X_test)[:, 1]

frac_pos_base,  mean_pred_base  = calibration_curve(y_test, prob_base,  n_bins=10)
frac_pos_calib, mean_pred_calib = calibration_curve(y_test, prob_calib, n_bins=10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Probability Calibration — Reliability Diagram",
             fontsize=14, fontweight="bold")

ax = axes[0]
ax.plot([0, 1], [0, 1], linestyle="--", color="#999999", label="Perfectly calibrated")
ax.plot(mean_pred_base,  frac_pos_base,  marker="o", color=PALETTE["coral"],
        linewidth=2,   label="Uncalibrated")
ax.plot(mean_pred_calib, frac_pos_calib, marker="s", color=PALETTE["teal"],
        linewidth=2,   label="Calibrated (isotonic)")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Reliability Diagram")
ax.legend()
ax.grid(True)

# Score histograms
ax2 = axes[1]
ax2.hist(prob_base[y_test == 0],  bins=40, alpha=0.6, color=PALETTE["teal"],
         label="Non-default (uncalib)",   density=True)
ax2.hist(prob_base[y_test == 1],  bins=40, alpha=0.6, color=PALETTE["coral"],
         label="Default (uncalib)",       density=True)
ax2.set_xlabel("Predicted probability")
ax2.set_ylabel("Density")
ax2.set_title("Score Distribution (Uncalibrated)")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
fig.savefig(REPORT_DIR / "c5_calibration.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved → data/08_reporting/c5_calibration.png")

# Use calibrated model from here on
FINAL_MODEL = CALIB_MODEL
y_prob      = FINAL_MODEL.predict_proba(X_test)[:, 1]
print(f"\nFinal model: CalibratedClassifierCV(isotonic)")
print(f"Test set ROC-AUC (calibrated): {roc_auc_score(y_test, y_prob):.4f}")

<a id="c6-threshold"></a>
## C6 — Decision Threshold Selection: Cost-Sensitive

### Verbraken et al. (2014) Cost Framework

The default classification threshold of 0.5 is appropriate only when the cost of a **false negative** (FN — a defaulter we let through) equals the cost of a **false positive** (FP — a good customer we reject).  In credit risk this is never the case.

Following **Verbraken et al. (2014)** (*"A Novel Profit Maximizing Metric for Measuring Classification Performance of Customer Churn Prediction Models"*), we define:

| Error Type | Business meaning | Cost |
|------------|-----------------|------|
| False Negative | Approve a defaulting loan | `c_fn = 10` |
| False Positive | Reject a creditworthy applicant | `c_fp = 1` |

Total expected cost at threshold *t*:

$$\text{Cost}(t) = c_{\text{fn}} \cdot FN(t) + c_{\text{fp}} \cdot FP(t)$$

We sweep *t* from 0.05 to 0.95 and select the threshold that **minimises** this cost.

In [ ]:
# ── Cost-sensitive threshold sweep ─────────────────────────────────────────────
C_FN = 10.0  # Cost of missing a defaulter
C_FP = 1.0   # Cost of rejecting a creditworthy customer

thresholds   = np.linspace(0.05, 0.95, 100)
total_costs  = []
f1_scores    = []

from sklearn.metrics import f1_score

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    # cm: [[TN, FP], [FN, TP]]
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (cm[0,0], 0, 0, 0)
    cost = C_FN * fn + C_FP * fp
    total_costs.append(cost)
    f1_scores.append(f1_score(y_test, y_pred_t, zero_division=0))

total_costs = np.array(total_costs)
f1_scores   = np.array(f1_scores)

best_idx  = np.argmin(total_costs)
THRESHOLD = thresholds[best_idx]

print(f"Optimal threshold (min cost): {THRESHOLD:.3f}")
print(f"  Min total cost            : {total_costs[best_idx]:.1f}")
print(f"  F1 at optimal threshold   : {f1_scores[best_idx]:.4f}")

In [ ]:
# ── Cost curve visualisation ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Decision Threshold Selection — Cost-Sensitive",
             fontsize=14, fontweight="bold")

# Cost curve
ax = axes[0]
ax.plot(thresholds, total_costs, color=PALETTE["dark_teal"], linewidth=2.5,
        label=f"Total cost (FN×{C_FN:.0f} + FP×{C_FP:.0f})")
ax.axvline(THRESHOLD, color=PALETTE["coral"], linestyle="--", linewidth=2,
           label=f"Optimal threshold = {THRESHOLD:.3f}")
ax.scatter([THRESHOLD], [total_costs[best_idx]], color=PALETTE["coral"],
           s=100, zorder=5)
ax.set_xlabel("Classification Threshold")
ax.set_ylabel("Total Expected Cost")
ax.set_title("Verbraken et al. (2014) Cost Curve")
ax.legend()
ax.yaxis.grid(True)
ax.set_axisbelow(True)

# F1 vs threshold
ax2 = axes[1]
ax2.plot(thresholds, f1_scores, color=PALETTE["teal"], linewidth=2.5, label="F1 Score")
ax2.axvline(THRESHOLD, color=PALETTE["coral"], linestyle="--", linewidth=2,
            label=f"Optimal threshold = {THRESHOLD:.3f}")
ax2.set_xlabel("Classification Threshold")
ax2.set_ylabel("F1 Score")
ax2.set_title("F1 Score vs Threshold")
ax2.legend()
ax2.yaxis.grid(True)
ax2.set_axisbelow(True)

plt.tight_layout()
fig.savefig(REPORT_DIR / "c6_threshold_selection.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved → data/08_reporting/c6_threshold_selection.png")
print(f"\nSelected decision threshold: {THRESHOLD:.3f}")

<a id="c7-evaluation"></a>
## C7 — Model Evaluation: Credit Scoring Metrics

Credit scoring models are evaluated using a suite of metrics that capture different aspects of discrimination and calibration:

| Metric | Formula / Notes |
|--------|-----------------|
| **ROC-AUC** | Area under ROC curve; probability that model ranks a defaulter above a non-defaulter |
| **Gini coefficient** | `2 × AUC − 1`; industry standard in credit risk (range: 0–1) |
| **KS statistic** | Max separation between default and non-default CDFs |
| **PR-AUC** | Area under Precision-Recall curve; better for imbalanced classes |
| **Brier score** | Mean squared error of probability forecasts; lower is better |
| **Log-loss** | Cross-entropy; penalises overconfident wrong predictions |

In [ ]:
# ── Compute metrics ────────────────────────────────────────────────────────────
y_pred = (y_prob >= THRESHOLD).astype(int)

roc_auc  = roc_auc_score(y_test, y_prob)
gini     = 2 * roc_auc - 1
pr_auc   = average_precision_score(y_test, y_prob)
brier    = brier_score_loss(y_test, y_prob)
logloss  = log_loss(y_test, y_prob)

# KS statistic
scores_pos = y_prob[y_test == 1]
scores_neg = y_prob[y_test == 0]
ks_stat, _  = ks_2samp(scores_pos, scores_neg)

metrics_df = pd.DataFrame([
    {"Metric": "ROC-AUC",    "Value": roc_auc,  "Higher is Better": True },
    {"Metric": "Gini",       "Value": gini,     "Higher is Better": True },
    {"Metric": "KS Stat",    "Value": ks_stat,  "Higher is Better": True },
    {"Metric": "PR-AUC",     "Value": pr_auc,   "Higher is Better": True },
    {"Metric": "Brier Score","Value": brier,    "Higher is Better": False},
    {"Metric": "Log-Loss",   "Value": logloss,  "Higher is Better": False},
])

print("Credit Scoring Metrics (Test Set):")
print(metrics_df.to_string(index=False))

In [ ]:
# ── 2×2 Evaluation panel ──────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
prec, rec, _= precision_recall_curve(y_test, y_prob)
cm          = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle("Model Evaluation — Credit Scoring Metrics",
             fontsize=15, fontweight="bold", y=1.01)

# ── ROC curve ──────────────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(fpr, tpr, color=PALETTE["teal"], linewidth=2.5,
        label=f"ROC-AUC = {roc_auc:.4f}   Gini = {gini:.4f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="#AAAAAA", label="Random")
ax.fill_between(fpr, tpr, alpha=0.08, color=PALETTE["teal"])
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend(fontsize=9)
ax.grid(True)

# ── PR curve ───────────────────────────────────────────────────────────────────
ax = axes[0, 1]
ax.plot(rec, prec, color=PALETTE["coral"], linewidth=2.5,
        label=f"PR-AUC = {pr_auc:.4f}")
baseline = y_test.mean()
ax.axhline(baseline, linestyle="--", color="#AAAAAA", label=f"Baseline = {baseline:.3f}")
ax.fill_between(rec, prec, alpha=0.08, color=PALETTE["coral"])
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend(fontsize=9)
ax.grid(True)

# ── Score distribution (KDE) ───────────────────────────────────────────────────
ax = axes[1, 0]
from scipy.stats import gaussian_kde

for label, color, name in [
    (0, PALETTE["teal"],  "Non-default (0)"),
    (1, PALETTE["coral"], "Default (1)"),
]:
    scores = y_prob[y_test == label]
    if len(scores) > 1:
        kde = gaussian_kde(scores, bw_method=0.3)
        x_range = np.linspace(0, 1, 300)
        ax.plot(x_range, kde(x_range), linewidth=2.5, color=color, label=name)
        ax.fill_between(x_range, kde(x_range), alpha=0.15, color=color)

ax.axvline(THRESHOLD, color=PALETTE["dark_teal"], linestyle="--", linewidth=2,
           label=f"Threshold = {THRESHOLD:.3f}")
ax.set_xlabel("Predicted Probability")
ax.set_ylabel("Density")
ax.set_title(f"Score Distribution by Class  (KS={ks_stat:.4f})")
ax.legend(fontsize=9)
ax.grid(True)

# ── Confusion matrix ───────────────────────────────────────────────────────────
ax = axes[1, 1]
im = ax.imshow(cm, interpolation="nearest",
               cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
thresh_cm = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f"{cm[i, j]:,}",
                ha="center", va="center", fontsize=13, fontweight="bold",
                color="white" if cm[i, j] > thresh_cm else "black")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred 0", "Pred 1"])
ax.set_yticklabels(["True 0", "True 1"])
ax.set_title(f"Confusion Matrix @ threshold={THRESHOLD:.3f}")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")

plt.tight_layout()
fig.savefig(REPORT_DIR / "c7_evaluation_panel.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved → data/08_reporting/c7_evaluation_panel.png")

In [ ]:
# ── Styled metrics table ────────────────────────────────────────────────────────
metrics_df[["Metric", "Value"]].style \
    .background_gradient(subset=["Value"], cmap="YlGn") \
    .format({"Value": "{:.4f}"}) \
    .set_caption("Credit Scoring Metrics — Test Set") \
    .set_properties(**{"font-size": "12px", "text-align": "center"})

<a id="c8-shap"></a>
## C8 — SHAP Explanations

### TreeExplainer for Global & Local Interpretability (Week 7)

**SHAP (SHapley Additive exPlanations)** decomposes each prediction into the contribution of each feature, grounded in cooperative game theory.  For tree-based models, `shap.TreeExplainer` computes exact SHAP values in polynomial time by exploiting the tree structure — far more efficient than the exponential brute-force approach.

SHAP values provide:
- **Global explanations** — which features matter most across all predictions (beeswarm, bar plot)
- **Local explanations** — why a specific individual received a high/low PD score (waterfall / force plots)
- **Interaction effects** — how pairs of features jointly influence predictions (dependence plots)

In [ ]:
# ── Extract the base estimator for SHAP (TreeExplainer needs a tree model) ─────
def extract_base_estimator(model):
    """Recursively unwrap CalibratedClassifierCV to get the raw tree model."""
    if hasattr(model, "calibrated_classifiers_"):
        return model.calibrated_classifiers_[0].estimator
    if hasattr(model, "estimator"):
        return extract_base_estimator(model.estimator)
    return model

BASE_TREE = extract_base_estimator(FINAL_MODEL)
print(f"Base tree model type: {type(BASE_TREE).__name__}")

# Sub-sample test set for SHAP (full set can be slow)
N_SHAP  = min(500, len(X_test))
rng     = np.random.default_rng(42)
shap_idx= rng.choice(len(X_test), size=N_SHAP, replace=False)
X_shap  = X_test.iloc[shap_idx].reset_index(drop=True)

explainer   = shap.TreeExplainer(BASE_TREE)
shap_values = explainer.shap_values(X_shap)

# For binary classifiers shap_values may be a list [class0, class1]
if isinstance(shap_values, list):
    sv = shap_values[1]   # class 1 (default)
else:
    sv = shap_values

print(f"SHAP values shape : {sv.shape}")
print(f"Samples used      : {N_SHAP}")

In [ ]:
# ── SHAP Beeswarm summary plot ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    sv, X_shap,
    plot_type="dot",
    max_display=15,
    show=False,
    color_bar=True,
)
plt.title("SHAP Beeswarm Summary — Feature Impact on Default Probability",
          fontsize=12, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(REPORT_DIR / "c8_shap_beeswarm.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved → data/08_reporting/c8_shap_beeswarm.png")

In [ ]:
# ── SHAP Bar plot (mean |SHAP|) ────────────────────────────────────────────────
mean_abs_shap = pd.Series(
    np.abs(sv).mean(axis=0),
    index=X_shap.columns
).sort_values(ascending=False)

top_n = 15
fig, ax = plt.subplots(figsize=(9, 6))
colors = [
    PALETTE["teal"] if i < 5 else
    PALETTE["dark_teal"] if i < 10 else
    PALETTE["coral"]
    for i in range(top_n)
]
bars = ax.barh(
    mean_abs_shap.head(top_n).index[::-1],
    mean_abs_shap.head(top_n).values[::-1],
    color=colors[::-1],
    edgecolor="white",
    linewidth=0.8,
)
for bar in bars:
    ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.4f}", va="center", ha="left", fontsize=8)
ax.set_xlabel("Mean |SHAP Value|")
ax.set_title("Top Feature Importances (SHAP)", fontweight="bold")
ax.xaxis.grid(True)
ax.set_axisbelow(True)
plt.tight_layout()
fig.savefig(REPORT_DIR / "c8_shap_bar.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved → data/08_reporting/c8_shap_bar.png")

print("\nTop 10 features by mean |SHAP|:")
print(mean_abs_shap.head(10).to_string())

In [ ]:
# ── SHAP Dependence plots for top-2 features ───────────────────────────────────
top2 = mean_abs_shap.head(2).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("SHAP Dependence Plots — Top 2 Features",
             fontsize=13, fontweight="bold")

for idx, (feat, ax) in enumerate(zip(top2, axes)):
    # interaction feature = second most important (or feature itself)
    interact_feat = top2[1 - idx]
    shap.dependence_plot(
        feat,
        sv,
        X_shap,
        interaction_index=interact_feat,
        ax=ax,
        show=False,
        alpha=0.6,
        dot_size=20,
    )
    ax.set_title(f"SHAP: {feat} (coloured by {interact_feat})", fontsize=10)
    ax.grid(True, alpha=0.5)

plt.tight_layout()
fig.savefig(REPORT_DIR / "c8_shap_dependence.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved → data/08_reporting/c8_shap_dependence.png")

<a id="c9-registry"></a>
## C9 — MLflow Model Registry

Once the champion model is validated, we log it to the **MLflow Model Registry** — a centralised catalogue that tracks model versions, lifecycle stages, and aliases.  This mirrors the Kedro pipeline's MLflow integration in `pipelines/model_train`.

The `"production"` alias enables downstream serving components (FastAPI in Part E) to load the current production model by name without hard-coding a version number.

In [ ]:
# ── Log final model to MLflow ──────────────────────────────────────────────────
MODEL_NAME = "HomeCredit"

with mlflow.start_run(run_name="final_model_registration") as run:
    # Log hyperparameters
    mlflow.log_param("champion",       CHAMPION_NAME)
    mlflow.log_param("calibration",    "isotonic")
    mlflow.log_param("threshold",      round(float(THRESHOLD), 4))
    mlflow.log_param("train_samples",  len(y_train_full))
    mlflow.log_param("test_samples",   len(y_test))

    # Log metrics
    mlflow.log_metric("test_roc_auc",  roc_auc)
    mlflow.log_metric("test_gini",     gini)
    mlflow.log_metric("test_ks_stat",  ks_stat)
    mlflow.log_metric("test_pr_auc",   pr_auc)
    mlflow.log_metric("test_brier",    brier)
    mlflow.log_metric("test_logloss",  logloss)

    # Log model artifact
    mlflow.sklearn.log_model(
        sk_model       = FINAL_MODEL,
        artifact_path  = "model",
        registered_model_name=MODEL_NAME,
        input_example  = X_test.head(5),
    )

    RUN_ID = run.info.run_id

print(f"Run ID    : {RUN_ID}")
print(f"Model     : {MODEL_NAME}")

In [ ]:
# ── Set 'production' alias on the latest version ───────────────────────────────
client = MlflowClient(tracking_uri=f"sqlite:///{MLFLOW_DB}")

# Get the latest version
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_v = max(int(v.version) for v in versions)

try:
    client.set_registered_model_alias(
        name=MODEL_NAME,
        alias="production",
        version=str(latest_v),
    )
    print(f"Alias 'production' set → {MODEL_NAME} v{latest_v}")
except Exception as e:
    # Older MLflow versions use transition_model_version_stage instead
    client.transition_model_version_stage(
        name=MODEL_NAME, version=str(latest_v), stage="Production", archive_existing_versions=True
    )
    print(f"Stage 'Production' set → {MODEL_NAME} v{latest_v}  (alias API not available: {e})")

In [ ]:
# ── Show how to load the production model ─────────────────────────────────────
print("Loading production model from registry …")
try:
    loaded_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/production")
    test_preds   = loaded_model.predict_proba(X_test.head(3))[:, 1]
    print(f"Loaded model type : {type(loaded_model).__name__}")
    print(f"Sample predictions (first 3): {test_preds.round(4)}")
except Exception as e:
    print(f"Load via alias failed ({e}) — trying version number.")
    loaded_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{latest_v}")
    test_preds   = loaded_model.predict_proba(X_test.head(3))[:, 1]
    print(f"Loaded model (v{latest_v}) — predictions: {test_preds.round(4)}")

print("\n✓  Model registry workflow complete.")

In [ ]:
# ── Print registered model info ────────────────────────────────────────────────
model_info = client.get_registered_model(MODEL_NAME)
print(f"Registered model   : {model_info.name}")
print(f"Description        : {model_info.description or 'N/A'}")
print(f"Latest versions    : {[v.version for v in versions]}")
print(f"Current production : v{latest_v}")

# Display version metadata as DataFrame
version_rows = []
for v in versions:
    version_rows.append({
        "Version" : v.version,
        "Status"  : v.status,
        "Stage"   : v.current_stage,
        "Run ID"  : v.run_id[:8] + "...",
        "Source"  : v.source.split("/")[-2] if "/" in (v.source or "") else v.source,
    })
pd.DataFrame(version_rows).style.set_caption("Registered Model Versions")

<a id="summary"></a>
## Summary

This notebook prototyped the full **Part C — Modeling** workflow for the Home Credit Default Risk MLOps pipeline.

### Key Outputs

| Section | Key Output | Kedro Pipeline Node |
|---------|-----------|---------------------|
| **C1 — Data Split** | Stratified 70/15/15 split, zero leakage | `data_split_node` in `pipelines/data_split` |
| **C2 — GridSearchCV** | Best CV AUC per family, MLflow run logs | `grid_search_node` in `pipelines/model_selection` |
| **C3 — Optuna TPE** | Best hyperparams from 20 TPE trials, importance ranking | `optuna_search_node` in `pipelines/model_selection` |
| **C4 — Compare** | Champion model selection table | `select_champion_node` in `pipelines/model_selection` |
| **C5 — Final Train** | `CalibratedClassifierCV(isotonic)` trained on full train | `train_final_model_node` in `pipelines/model_train` |
| **C6 — Threshold** | Optimal threshold via Verbraken et al. cost framework | `select_threshold_node` in `pipelines/model_train` |
| **C7 — Evaluation** | ROC-AUC, Gini, KS, PR-AUC, Brier, Log-loss panel | `evaluate_model_node` in `pipelines/model_train` |
| **C8 — SHAP** | Beeswarm, bar, dependence plots; top-10 features | `shap_explain_node` in `pipelines/model_train` |
| **C9 — Registry** | Artifact logged, `production` alias assigned | `register_model_node` in `pipelines/model_train` |

### Next Steps

- **Part D — Data Drift Detection**: integrate [Evidently AI](https://evidentlyai.com) to monitor feature distributions and model performance drift over time, generating HTML reports and JSON metrics compatible with the Kedro pipeline
- **Part E — Model Serving**: wrap the production model in a [FastAPI](https://fastapi.tiangolo.com) REST endpoint with `/predict` and `/health` routes, Dockerise, and deploy with a CI/CD pipeline
- **Part F — Orchestration**: migrate the Kedro pipeline to Apache Airflow or Prefect for scheduled retraining triggered by drift alerts

---
*Group 1 — MLOps, MSc Data Science & Advanced Analytics, NOVA IMS 2025/2026*